## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        self.W1 = tf.Variable(tf.random.normal(shape=[28 * 28, 100], stddev=0.1), name='W1')
        self.b1 = tf.Variable(tf.zeros(shape=[100]), name='b1')
        self.W2 = tf.Variable(tf.random.normal(shape=[100, 10], stddev=0.1), name='W2')
        self.b2 = tf.Variable(tf.zeros(shape=[10]), name='b2')

    def __call__(self, x):
        x = tf.reshape(x, [-1, 28 * 28])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [ ]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1, output_type=tf.int64)
    labels = tf.cast(labels, tf.int64)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [8]:
train_data, test_data = mnist_dataset()

x_train = tf.constant(train_data[0], dtype=tf.float32)
y_train = tf.constant(train_data[1], dtype=tf.int64)
x_test = tf.constant(test_data[0], dtype=tf.float32)
y_test = tf.constant(test_data[1], dtype=tf.int64)

batch_size = 128
epochs = 10
num_train = x_train.shape[0]

for epoch in range(epochs):
    idx = tf.random.shuffle(tf.range(num_train))
    x_train_epoch = tf.gather(x_train, idx)
    y_train_epoch = tf.gather(y_train, idx)

    epoch_loss = []
    epoch_acc = []
    for i in range(0, num_train, batch_size):
        xb = x_train_epoch[i:i + batch_size]
        yb = y_train_epoch[i:i + batch_size]
        loss, accuracy = train_one_step(model, optimizer, xb, yb)
        epoch_loss.append(loss.numpy())
        epoch_acc.append(accuracy.numpy())

    test_loss, test_acc = test(model, x_test, y_test)
    print('epoch', epoch,
          ': train loss', float(np.mean(epoch_loss)),
          '; train accuracy', float(np.mean(epoch_acc)),
          '; test loss', test_loss.numpy(),
          '; test accuracy', test_acc.numpy())

epoch 0 : train loss 0.2425052970647812 ; train accuracy 0.9313033223152161 ; test loss 0.23566909 ; test accuracy 0.9311
epoch 1 : train loss 0.23805244266986847 ; train accuracy 0.9320084452629089 ; test loss 0.23209941 ; test accuracy 0.9319
epoch 2 : train loss 0.23374152183532715 ; train accuracy 0.9335909485816956 ; test loss 0.2296346 ; test accuracy 0.9326
epoch 3 : train loss 0.22957661747932434 ; train accuracy 0.9343905448913574 ; test loss 0.22616372 ; test accuracy 0.9337
epoch 4 : train loss 0.22564715147018433 ; train accuracy 0.9355843663215637 ; test loss 0.2217335 ; test accuracy 0.9349
epoch 5 : train loss 0.2218593806028366 ; train accuracy 0.9367837309837341 ; test loss 0.21789692 ; test accuracy 0.9361
epoch 6 : train loss 0.21815337240695953 ; train accuracy 0.9378942847251892 ; test loss 0.21503137 ; test accuracy 0.9366
epoch 7 : train loss 0.21457616984844208 ; train accuracy 0.9391046762466431 ; test loss 0.21156186 ; test accuracy 0.9371
epoch 8 : train loss